
# Problem B — Synthetic Event Log Evaluation
## Fidelity · Utility · Privacy, generator-seed based evaluation

This notebook rewrites the previous version to match the guideline structure and the intended seed protocol.

Core protocol:

1. **Generator seed protocol**: each synthetic log file is treated as one generator seed, e.g. `ProcessGAN_seed41.csv` … `seed45.csv`. Metrics are computed per `(generator, seed)` and then aggregated as `mean ± std`.
2. **Fidelity**: real train log vs synthetic log.
   - Activity frequency KL-divergence ↓
   - Trace variant set Jaccard similarity ↑
   - Case-duration Wasserstein distance ↓
3. **Utility**: TSTR.
   - Train predictor on synthetic log.
   - Test predictor on real test log.
   - Compare with Train-on-Real Test-on-Real baseline.
   - Targets: readmission, LOS exceedance, admission conversion.
4. **Privacy**: ML-Leaks adapted to synthetic event logs.
   - Shadow generator → shadow synthetic log.
   - Shadow defender → posterior features.
   - MLP attack classifier: member vs non-member.
   - Target defender trained on each synthetic seed → MIA AUC.

Important correction:

- The notebook does **not** fabricate clinical labels for synthetic logs. If a target label cannot be derived from a synthetic log or mapped by `stay_id`, the corresponding Utility/MIA cell is marked `NA` with an explicit reason.
- Full-trace leakage features such as total duration, full trace length, and last activity are excluded from Utility/Privacy defender features.


In [1]:

# ============================================================
# 0. Imports
# ============================================================
import os, re, json, math, zipfile, warnings, shutil
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from scipy.special import rel_entr
from scipy.stats import wasserstein_distance

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)
print('Imports loaded')


Imports loaded


In [2]:

# ============================================================
# 1. Configuration
# ============================================================
DATA_ZIP = Path('./data.zip')
DATA_DIR = Path('./data')
OUT_DIR = Path('./results_problemB_seeded_corrected')
OUT_DIR.mkdir(parents=True, exist_ok=True)

CASE_COL = 'stay_id'
ACT_COL = 'activity'
TIME_COL = 'timestamps'

GENERATORS = ['ProcessGAN', 'PALSYN', 'Rule_based']
EXPECTED_SEEDS = [41, 42, 43, 44, 45]
RANDOM_STATE = 42

# Prediction targets required by the guideline.
TARGETS = {
    'readmission_30d': '재입원 예측',
    'los_over_4h': '체류시간 초과',
    'admission_conversion': '입원 전환',
}

# Prefix length used for downstream prediction. Full trace outcome leakage is excluded.
PREFIX_EVENTS = 3

# MIA criterion in the guideline.
MIA_VULNERABLE_AUC = 0.60

print('Configuration ready')


Configuration ready


In [3]:

# ============================================================
# 2. Extract and discover files
# ============================================================
if not DATA_DIR.exists():
    if DATA_ZIP.exists():
        with zipfile.ZipFile(DATA_ZIP, 'r') as zf:
            zf.extractall(DATA_DIR)
        print(f'Extracted {DATA_ZIP} -> {DATA_DIR}')
    else:
        raise FileNotFoundError('data.zip not found and ./data directory does not exist.')
else:
    print('Using existing ./data directory')

print('Top-level files/folders:')
for p in sorted(DATA_DIR.glob('*')):
    print(' -', p)


Using existing ./data directory
Top-level files/folders:
 - data\data.zip
 - data\PALSYN
 - data\ProcessGAN
 - data\Real
 - data\Rule_based


In [4]:

# ============================================================
# 3. Loading and standardization
# ============================================================
def read_table(path: Path) -> pd.DataFrame:
    path = Path(path)
    if path.suffix.lower() == '.csv':
        return pd.read_csv(path)
    if path.suffix.lower() in ['.parquet', '.pq']:
        return pd.read_parquet(path)
    if path.suffix.lower() in ['.xlsx', '.xls']:
        return pd.read_excel(path)
    raise ValueError(f'Unsupported file: {path}')


def standardize_event_log(df: pd.DataFrame, source_name: str = '') -> pd.DataFrame:
    """Normalize common event-log column names.
    Supports normal CSV style and XES-export style.
    """
    df = df.copy()
    rename = {}
    if 'case:concept:name' in df.columns and CASE_COL not in df.columns:
        rename['case:concept:name'] = CASE_COL
    if 'concept:name' in df.columns and ACT_COL not in df.columns:
        rename['concept:name'] = ACT_COL
    if 'time:timestamp' in df.columns and TIME_COL not in df.columns:
        rename['time:timestamp'] = TIME_COL
    if 'timestamp' in df.columns and TIME_COL not in df.columns:
        rename['timestamp'] = TIME_COL
    if rename:
        df = df.rename(columns=rename)

    missing = [c for c in [CASE_COL, ACT_COL] if c not in df.columns]
    if missing:
        raise ValueError(f'{source_name}: missing required columns {missing}. Columns={df.columns.tolist()}')

    if TIME_COL not in df.columns:
        df[TIME_COL] = pd.NaT
    df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors='coerce')
    df[CASE_COL] = df[CASE_COL].astype(str)
    df[ACT_COL] = df[ACT_COL].astype(str)
    df = df.sort_values([CASE_COL, TIME_COL], na_position='last').reset_index(drop=True)
    return df


def find_real_file(kind: str) -> Path:
    """Find real train/test/val file from common locations."""
    candidates = []
    patterns = [f'*{kind}*.csv', f'*{kind}*.parquet', f'*{kind}*.xlsx']
    for root in [DATA_DIR, DATA_DIR/'Real', DATA_DIR/'real']:
        if root.exists():
            for pat in patterns:
                candidates.extend(root.glob(pat))
    # Prefer mimicel_train.csv style over synthetic-looking files.
    candidates = [p for p in candidates if not any(g.lower() in str(p).lower() for g in ['processgan','palsyn','rule_based','synthetic'])]
    if not candidates:
        raise FileNotFoundError(f'Cannot find real {kind} file under {DATA_DIR}')
    candidates = sorted(candidates, key=lambda p: (len(str(p)), str(p)))
    return candidates[0]

TRAIN_PATH = find_real_file('train')
TEST_PATH = find_real_file('test')
VAL_PATH = None
try:
    VAL_PATH = find_real_file('val')
except Exception:
    pass

real_train_raw = standardize_event_log(read_table(TRAIN_PATH), TRAIN_PATH.name)
real_test_raw = standardize_event_log(read_table(TEST_PATH), TEST_PATH.name)
real_val_raw = standardize_event_log(read_table(VAL_PATH), VAL_PATH.name) if VAL_PATH else None

print('Real train:', TRAIN_PATH, real_train_raw.shape)
print('Real test :', TEST_PATH, real_test_raw.shape)
if VAL_PATH:
    print('Real val  :', VAL_PATH, real_val_raw.shape)
print('Real train columns:', real_train_raw.columns.tolist())


Real train: data\Real\mimicel_train.csv (16176, 31)
Real test : data\Real\mimicel_test.csv (4659, 31)
Real val  : data\Real\mimicel_val.csv (2292, 31)
Real train columns: ['stay_id', 'subject_id', 'hadm_id', 'timestamps', 'activity', 'gender', 'race', 'arrival_transport', 'disposition', 'seq_num', 'icd_code', 'icd_version', 'icd_title', 'temperature', 'heartrate', 'resprate', 'o2sat', 'sbp', 'dbp', 'pain', 'acuity', 'chiefcomplaint', 'rhythm', 'name', 'gsn', 'ndc', 'etc_rn', 'etccode', 'etcdescription', 'med_rn', 'gsn_rn']


In [5]:

# ============================================================
# 4. Discover synthetic logs by generator and seed
# ============================================================
def infer_generator(path: Path) -> str:
    s = str(path).lower()
    if 'processgan' in s or 'process_gan' in s:
        return 'ProcessGAN'
    if 'palsyn' in s:
        return 'PALSYN'
    if 'rule_based' in s or 'rule-based' in s or 'rulebased' in s:
        return 'Rule_based'
    return 'Unknown'


def infer_seed(path: Path, fallback_idx=None):
    s = path.stem.lower()
    patterns = [r'seed[_\- ]?(\d+)', r'[_\-](\d{2,4})[_\-]?', r'(\d{2,4})']
    for pat in patterns:
        hits = re.findall(pat, s)
        if hits:
            # Prefer expected seeds if present.
            ints = [int(x) for x in hits]
            for v in ints:
                if v in EXPECTED_SEEDS:
                    return v
            return ints[-1]
    return fallback_idx


def discover_synthetic_logs(data_dir: Path):
    rows = []
    # Search only under generator folders when present; otherwise recursively in data_dir.
    search_roots = []
    for gen in GENERATORS:
        for folder_name in [gen, gen.lower(), gen.replace('_','-'), gen.replace('_','')]:
            p = data_dir / folder_name
            if p.exists():
                search_roots.append(p)
    if not search_roots:
        search_roots = [data_dir]

    exts = ['*.csv','*.parquet','*.pq','*.xlsx','*.xls']
    paths = []
    for root in search_roots:
        for ext in exts:
            paths.extend(root.rglob(ext))

    # Exclude real train/test/val and outputs.
    real_names = {TRAIN_PATH.name, TEST_PATH.name}
    if VAL_PATH: real_names.add(VAL_PATH.name)
    paths = [p for p in paths if p.name not in real_names and 'real' not in str(p.parent).lower()]

    grouped_idx = defaultdict(int)
    for p in sorted(set(paths)):
        gen = infer_generator(p)
        if gen == 'Unknown':
            # Use parent folder if exact generator folder was used.
            parent = p.parent.name.lower()
            if 'process' in parent:
                gen = 'ProcessGAN'
            elif 'palsyn' in parent:
                gen = 'PALSYN'
            elif 'rule' in parent:
                gen = 'Rule_based'
        if gen == 'Unknown':
            continue
        grouped_idx[gen] += 1
        seed = infer_seed(p, fallback_idx=grouped_idx[gen])
        rows.append({'generator': gen, 'seed': int(seed), 'path': p})
    return pd.DataFrame(rows).drop_duplicates(['generator','seed','path']).sort_values(['generator','seed','path'])

synthetic_files = discover_synthetic_logs(DATA_DIR)
if synthetic_files.empty:
    raise FileNotFoundError('No synthetic log files discovered. Expected folders: data/ProcessGAN, data/PALSYN, data/Rule_based')

print('Discovered synthetic files:')
display(synthetic_files)

# Warn if expected 5 seeds are missing.
for gen in GENERATORS:
    seeds = sorted(synthetic_files.loc[synthetic_files.generator==gen, 'seed'].unique().tolist())
    print(f'{gen}: seeds={seeds}, n={len(seeds)}')


Discovered synthetic files:


,generator,seed,path
0,PALSYN,41,data\PALSYN\41mimicel_Sheet1.csv
1,PALSYN,42,data\PALSYN\42mimicel_Sheet1.csv
2,PALSYN,43,data\PALSYN\43mimicel_Sheet1.csv
3,PALSYN,44,data\PALSYN\44mimicel_Sheet1.csv
4,PALSYN,45,data\PALSYN\45mimicel_Sheet1.csv
5,ProcessGAN,41,data\ProcessGAN\processgan_restored_synthetic_...
6,ProcessGAN,42,data\ProcessGAN\processgan_restored_synthetic_...
7,ProcessGAN,43,data\ProcessGAN\processgan_restored_synthetic_...
8,ProcessGAN,44,data\ProcessGAN\processgan_restored_synthetic_...
9,ProcessGAN,45,data\ProcessGAN\processgan_restored_synthetic_...


ProcessGAN: seeds=[41, 42, 43, 44, 45], n=5
PALSYN: seeds=[41, 42, 43, 44, 45], n=5
Rule_based: seeds=[41, 42, 43, 44, 45], n=5


In [6]:

# ============================================================
# 5. Case-level table, labels, and leakage-safe prefix features
# ============================================================
def _safe_mode(s):
    s = s.dropna()
    if len(s) == 0:
        return np.nan
    m = s.mode()
    return m.iloc[0] if len(m) else s.iloc[0]


def build_case_base(df: pd.DataFrame, dataset_name='') -> pd.DataFrame:
    """Build case-level information and prefix-only features.
    Full-trace statistics are retained for labels/fidelity, but excluded from model features later.
    """
    rows = []
    df = df.sort_values([CASE_COL, TIME_COL], na_position='last')
    has_subject = 'subject_id' in df.columns
    for cid, g in df.groupby(CASE_COL, sort=False):
        acts = g[ACT_COL].astype(str).tolist()
        times = pd.to_datetime(g[TIME_COL], errors='coerce')
        case_start = times.min() if times.notna().any() else pd.NaT
        case_end = times.max() if times.notna().any() else pd.NaT
        duration_h = (case_end - case_start).total_seconds()/3600 if pd.notna(case_start) and pd.notna(case_end) else np.nan
        prefix = acts[:PREFIX_EVENTS]
        row = {
            CASE_COL: str(cid),
            'dataset': dataset_name,
            'trace_variant': '>>'.join(acts),
            'case_start': case_start,
            'case_end': case_end,
            'n_events_full': len(acts),
            'duration_hours_full': duration_h,
            'first_activity': acts[0] if acts else np.nan,
            'last_activity': acts[-1] if acts else np.nan,
        }
        if has_subject:
            row['subject_id'] = _safe_mode(g['subject_id'].astype(str))
        # Prefix activity positions.
        for i in range(PREFIX_EVENTS):
            row[f'prefix_act_{i+1}'] = prefix[i] if i < len(prefix) else '__PAD__'
        row['prefix_len'] = len(prefix)
        # Prefix counts only, not full counts.
        for a, cnt in Counter(prefix).items():
            row[f'prefix_count__{a}'] = cnt
        rows.append(row)
    out = pd.DataFrame(rows)
    count_cols = [c for c in out.columns if c.startswith('prefix_count__')]
    if count_cols:
        out[count_cols] = out[count_cols].fillna(0)
    return out


def add_real_labels(case_df: pd.DataFrame) -> pd.DataFrame:
    """Create the three target labels when they can be derived from real case metadata.
    No synthetic-only proxy labels are fabricated.
    """
    df = case_df.copy()
    # LOS > 4 hours can be derived from event timestamps.
    if 'duration_hours_full' in df.columns:
        df['los_over_4h'] = np.where(df['duration_hours_full'].notna(), (df['duration_hours_full'] > 4).astype(int), np.nan)
    else:
        df['los_over_4h'] = np.nan

    # Readmission within 30 days: requires subject_id and case_start.
    df['readmission_30d'] = np.nan
    if 'subject_id' in df.columns and 'case_start' in df.columns:
        tmp = df[[CASE_COL, 'subject_id', 'case_start']].copy()
        tmp = tmp.dropna(subset=['subject_id', 'case_start']).sort_values(['subject_id','case_start'])
        labels = {}
        for sid, g in tmp.groupby('subject_id'):
            starts = g['case_start'].tolist()
            ids = g[CASE_COL].tolist()
            for i, cid in enumerate(ids):
                if i + 1 < len(ids):
                    delta = (starts[i+1] - starts[i]).total_seconds() / (24*3600)
                    labels[cid] = int(0 < delta <= 30)
                else:
                    labels[cid] = 0
        df['readmission_30d'] = df[CASE_COL].map(labels)

    # Admission conversion: use explicit clinical columns if they exist. Do not infer from activity names by default.
    df['admission_conversion'] = np.nan
    candidate_cols = [c for c in df.columns if c.lower() in ['admission_conversion','admitted','hospitalized','is_admitted','ed_to_inpatient','inpatient_admission']]
    # Case table normally will not have these unless preserved. This is kept for compatibility.
    if candidate_cols:
        col = candidate_cols[0]
        df['admission_conversion'] = pd.to_numeric(df[col], errors='coerce')
    return df


def transfer_labels_by_case_id(syn_cases: pd.DataFrame, real_label_cases: pd.DataFrame) -> pd.DataFrame:
    """If synthetic case ids are copied from real cases, transfer labels by stay_id.
    Otherwise labels remain as originally derived from synthetic metadata.
    """
    out = syn_cases.copy()
    label_map = real_label_cases.set_index(CASE_COL)[list(TARGETS.keys())].to_dict(orient='index')
    for target in TARGETS:
        if target not in out.columns:
            out[target] = np.nan
        mapped = out[CASE_COL].map(lambda x: label_map.get(str(x), {}).get(target, np.nan))
        out[target] = out[target].where(out[target].notna(), mapped)
    return out

real_train_cases = add_real_labels(build_case_base(real_train_raw, 'real_train'))
real_test_cases = add_real_labels(build_case_base(real_test_raw, 'real_test'))
real_val_cases = add_real_labels(build_case_base(real_val_raw, 'real_val')) if real_val_raw is not None else None

print('Real train cases:', real_train_cases.shape)
print('Real test cases :', real_test_cases.shape)
display(real_train_cases[[CASE_COL]+list(TARGETS)].head())
print('Label availability in real_train:')
for t in TARGETS:
    y = real_train_cases[t]
    print(t, 'n=', int(y.notna().sum()), 'classes=', sorted(y.dropna().unique().tolist()), 'rate=', y.mean(skipna=True))


Real train cases: (899, 23)
Real test cases : (258, 23)


,stay_id,readmission_30d,los_over_4h,admission_conversion
0,30011360,0,0.0,NaN
1,30019063,0,1.0,NaN
2,30019263,0,1.0,NaN
3,30026761,0,1.0,NaN
4,30030013,0,1.0,NaN


Label availability in real_train:
readmission_30d n= 899 classes= [0] rate= 0.0
los_over_4h n= 899 classes= [0.0, 1.0] rate= 0.6840934371523916
admission_conversion n= 0 classes= [] rate= nan


In [7]:

# ============================================================
# 6. Load synthetic logs and build case tables by generator seed
# ============================================================
syn_logs = {}
syn_case_tables = {}
load_errors = []
for _, row in synthetic_files.iterrows():
    gen, seed, path = row['generator'], int(row['seed']), Path(row['path'])
    try:
        log = standardize_event_log(read_table(path), f'{gen}_seed{seed}')
        cases = build_case_base(log, f'{gen}_seed{seed}')
        cases = add_real_labels(cases)  # labels derivable from synthetic timestamps, e.g. los_over_4h
        cases = transfer_labels_by_case_id(cases, real_train_cases)  # if synthetic ids preserve real ids
        syn_logs[(gen, seed)] = log
        syn_case_tables[(gen, seed)] = cases
    except Exception as e:
        load_errors.append({'generator': gen, 'seed': seed, 'path': str(path), 'error': repr(e)})

if load_errors:
    print('[WARN] Some synthetic files failed to load:')
    display(pd.DataFrame(load_errors))

print('Loaded synthetic logs:', len(syn_logs))
for key, df in syn_logs.items():
    print(key, 'events=', len(df), 'cases=', syn_case_tables[key].shape[0])


Loaded synthetic logs: 15
('PALSYN', 41) events= 6000 cases= 813
('PALSYN', 42) events= 6363 cases= 791
('PALSYN', 43) events= 6228 cases= 812
('PALSYN', 44) events= 6011 cases= 816
('PALSYN', 45) events= 6374 cases= 820
('ProcessGAN', 41) events= 10355 cases= 899
('ProcessGAN', 42) events= 19578 cases= 897
('ProcessGAN', 43) events= 22376 cases= 899
('ProcessGAN', 44) events= 17590 cases= 897
('ProcessGAN', 45) events= 15192 cases= 899
('Rule_based', 41) events= 16564 cases= 899
('Rule_based', 42) events= 16988 cases= 899
('Rule_based', 43) events= 15814 cases= 899
('Rule_based', 44) events= 16326 cases= 899
('Rule_based', 45) events= 15703 cases= 899


In [8]:

# ============================================================
# 7. Label availability check
# ============================================================
label_rows = []
for (gen, seed), cases in syn_case_tables.items():
    for target in TARGETS:
        y = cases[target] if target in cases.columns else pd.Series(dtype=float)
        label_rows.append({
            'generator': gen,
            'seed': seed,
            'target': target,
            'target_korean': TARGETS[target],
            'n_labeled': int(y.notna().sum()),
            'n_cases': int(len(cases)),
            'positive_rate': float(y.mean(skipna=True)) if y.notna().sum() else np.nan,
            'n_classes': int(y.dropna().nunique()) if y.notna().sum() else 0,
            'classes': ','.join(map(str, sorted(y.dropna().unique().tolist()))) if y.notna().sum() else ''
        })
label_availability = pd.DataFrame(label_rows).sort_values(['generator','seed','target'])
label_availability.to_csv(OUT_DIR/'label_availability_check.csv', index=False)
display(label_availability)


,generator,seed,target,target_korean,n_labeled,n_cases,positive_rate,n_classes,classes
2,PALSYN,41,admission_conversion,입원 전환,0,813,NaN,0,
1,PALSYN,41,los_over_4h,체류시간 초과,813,813,0.682657,2,"0.0,1.0"
0,PALSYN,41,readmission_30d,재입원 예측,0,813,NaN,0,
5,PALSYN,42,admission_conversion,입원 전환,0,791,NaN,0,
4,PALSYN,42,los_over_4h,체류시간 초과,791,791,0.664981,2,"0.0,1.0"
3,PALSYN,42,readmission_30d,재입원 예측,0,791,NaN,0,
8,PALSYN,43,admission_conversion,입원 전환,0,812,NaN,0,
7,PALSYN,43,los_over_4h,체류시간 초과,812,812,0.692118,2,"0.0,1.0"
6,PALSYN,43,readmission_30d,재입원 예측,0,812,NaN,0,
11,PALSYN,44,admission_conversion,입원 전환,0,816,NaN,0,


In [9]:

# ============================================================
# 8. Fidelity metrics by generator seed
# ============================================================
def activity_distribution(df):
    return df[ACT_COL].astype(str).value_counts(normalize=True)


def activity_kl(real_df, syn_df, eps=1e-9):
    p, q = activity_distribution(real_df), activity_distribution(syn_df)
    acts = sorted(set(p.index) | set(q.index))
    pv = np.array([p.get(a,0.0) for a in acts]) + eps
    qv = np.array([q.get(a,0.0) for a in acts]) + eps
    pv = pv / pv.sum()
    qv = qv / qv.sum()
    return float(np.sum(rel_entr(pv, qv)))


def trace_variants(df):
    variants = []
    for _, g in df.sort_values([CASE_COL,TIME_COL], na_position='last').groupby(CASE_COL):
        variants.append('>>'.join(g[ACT_COL].astype(str).tolist()))
    return set(variants)


def variant_jaccard(real_df, syn_df):
    a, b = trace_variants(real_df), trace_variants(syn_df)
    if not a and not b:
        return np.nan
    return float(len(a & b) / len(a | b)) if len(a | b) else np.nan


def case_durations(df):
    vals = []
    for _, g in df.groupby(CASE_COL):
        t = pd.to_datetime(g[TIME_COL], errors='coerce')
        if t.notna().sum() >= 2:
            vals.append((t.max()-t.min()).total_seconds()/3600.0)
    return np.array(vals, dtype=float)


def duration_wasserstein(real_df, syn_df):
    r = case_durations(real_df)
    s = case_durations(syn_df)
    if len(r) == 0 or len(s) == 0:
        return np.nan
    return float(wasserstein_distance(r, s))

fidelity_rows = []
for (gen, seed), syn_df in syn_logs.items():
    fidelity_rows.append({
        'generator': gen,
        'seed': seed,
        'activity_kl_divergence': activity_kl(real_train_raw, syn_df),
        'trace_variant_jaccard': variant_jaccard(real_train_raw, syn_df),
        'duration_wasserstein_hours': duration_wasserstein(real_train_raw, syn_df),
    })

fidelity_by_seed = pd.DataFrame(fidelity_rows).sort_values(['generator','seed'])
fidelity_by_seed.to_csv(OUT_DIR/'fidelity_by_seed.csv', index=False)
display(fidelity_by_seed)


,generator,seed,activity_kl_divergence,trace_variant_jaccard,duration_wasserstein_hours
0,PALSYN,41,0.221271,0.001387,29588.675913
1,PALSYN,42,0.208383,0.001434,32411.230795
2,PALSYN,43,0.239477,0.000683,23931.993584
3,PALSYN,44,0.224584,0.000686,20354.663045
4,PALSYN,45,0.243367,0.000000,22837.476805
5,ProcessGAN,41,0.180808,0.001789,7.662370
6,ProcessGAN,42,0.013660,0.000000,7.574599
7,ProcessGAN,43,0.032878,0.000000,7.572976
8,ProcessGAN,44,0.087908,0.000000,7.592458
9,ProcessGAN,45,0.875203,0.000000,7.566301


In [10]:

# ============================================================
# 9. Utility: TSTR AUC by generator seed and target
# ============================================================
def get_model_feature_columns(case_df: pd.DataFrame):
    """Use only leakage-safe early-prefix features.
    Exclude full-trace properties and labels.
    """
    exclude = {
        CASE_COL, 'dataset', 'trace_variant', 'case_start', 'case_end', 'subject_id',
        'n_events_full', 'duration_hours_full', 'last_activity'
    } | set(TARGETS.keys())
    return [c for c in case_df.columns if c not in exclude]


def align_feature_frames(train_df: pd.DataFrame, test_df: pd.DataFrame, feature_cols=None):
    if feature_cols is None:
        feature_cols = sorted(set(get_model_feature_columns(train_df)) | set(get_model_feature_columns(test_df)))
    tr = train_df.reindex(columns=feature_cols).copy()
    te = test_df.reindex(columns=feature_cols).copy()
    return tr, te, feature_cols


def make_preprocessor(X: pd.DataFrame):
    cat_cols = [c for c in X.columns if X[c].dtype == 'object' or str(X[c].dtype).startswith('category')]
    num_cols = [c for c in X.columns if c not in cat_cols]
    try:
        ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        ohe = OneHotEncoder(handle_unknown='ignore', sparse=False)
    return ColumnTransformer([
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_cols),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', ohe)]), cat_cols),
    ], remainder='drop')


def make_predictor(seed=RANDOM_STATE):
    # RandomForest is robust when xgboost is not installed and handles small/imbalanced data reasonably.
    return RandomForestClassifier(n_estimators=300, random_state=seed, class_weight='balanced_subsample', min_samples_leaf=2, n_jobs=-1)


def evaluate_binary_predictor(train_cases, test_cases, target, seed=RANDOM_STATE):
    """Train on train_cases and evaluate on test_cases using AUC/ACC/F1.
    Returns skip reason if labels are unavailable or single-class.
    """
    if target not in train_cases.columns or target not in test_cases.columns:
        return {'status': 'skipped', 'reason': 'target_missing'}
    train = train_cases[train_cases[target].notna()].copy()
    test = test_cases[test_cases[target].notna()].copy()
    if len(train) < 20 or len(test) < 20:
        return {'status': 'skipped', 'reason': f'too_few_labeled_train_or_test train={len(train)} test={len(test)}'}
    if train[target].nunique() < 2:
        return {'status': 'skipped', 'reason': 'single_class_train'}
    if test[target].nunique() < 2:
        return {'status': 'skipped', 'reason': 'single_class_test'}
    X_train, X_test, feature_cols = align_feature_frames(train, test)
    y_train = train[target].astype(int).values
    y_test = test[target].astype(int).values
    pipe = Pipeline([('prep', make_preprocessor(X_train)), ('clf', make_predictor(seed))])
    pipe.fit(X_train, y_train)
    if hasattr(pipe.named_steps['clf'], 'predict_proba'):
        score = pipe.predict_proba(X_test)[:,1]
    else:
        score = pipe.decision_function(X_test)
    pred = (score >= 0.5).astype(int)
    return {
        'status': 'ok',
        'auc': float(roc_auc_score(y_test, score)),
        'accuracy': float(accuracy_score(y_test, pred)),
        'f1': float(f1_score(y_test, pred, zero_division=0)),
        'n_train': int(len(train)),
        'n_test': int(len(test)),
        'pos_rate_train': float(np.mean(y_train)),
        'pos_rate_test': float(np.mean(y_test)),
        'n_features': int(len(feature_cols)),
        'reason': ''
    }

# Train-on-Real baseline per target.
baseline_rows = []
for target, target_ko in TARGETS.items():
    res = evaluate_binary_predictor(real_train_cases, real_test_cases, target, seed=RANDOM_STATE)
    baseline_rows.append({'target': target, 'target_korean': target_ko, **{f'baseline_{k}': v for k,v in res.items()}})
baseline_df = pd.DataFrame(baseline_rows)
baseline_df.to_csv(OUT_DIR/'utility_train_on_real_baseline.csv', index=False)
display(baseline_df)

utility_rows = []
for (gen, seed), syn_cases in syn_case_tables.items():
    for target, target_ko in TARGETS.items():
        res = evaluate_binary_predictor(syn_cases, real_test_cases, target, seed=seed)
        base_res = baseline_df.loc[baseline_df.target==target].iloc[0].to_dict()
        baseline_auc = base_res.get('baseline_auc', np.nan) if base_res.get('baseline_status') == 'ok' else np.nan
        utility_auc = res.get('auc', np.nan) if res.get('status') == 'ok' else np.nan
        gap = baseline_auc - utility_auc if pd.notna(baseline_auc) and pd.notna(utility_auc) else np.nan
        utility_rows.append({
            'generator': gen,
            'seed': seed,
            'target': target,
            'target_korean': target_ko,
            'tstr_auc': utility_auc,
            'train_on_real_auc': baseline_auc,
            'utility_gap': gap,
            'tstr_accuracy': res.get('accuracy', np.nan),
            'tstr_f1': res.get('f1', np.nan),
            'status': res.get('status'),
            'reason': res.get('reason',''),
            'n_synthetic_train': res.get('n_train', np.nan),
            'n_real_test': res.get('n_test', np.nan),
            'pos_rate_synthetic_train': res.get('pos_rate_train', np.nan),
            'pos_rate_real_test': res.get('pos_rate_test', np.nan),
        })

utility_by_seed = pd.DataFrame(utility_rows).sort_values(['generator','seed','target'])
utility_by_seed.to_csv(OUT_DIR/'utility_tstr_by_seed.csv', index=False)
display(utility_by_seed)


,target,target_korean,baseline_status,baseline_reason,baseline_auc,baseline_accuracy,baseline_f1,baseline_n_train,baseline_n_test,baseline_pos_rate_train,baseline_pos_rate_test,baseline_n_features
0,readmission_30d,재입원 예측,skipped,single_class_train,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,los_over_4h,체류시간 초과,ok,,0.639732,0.616279,0.668896,899.0,258.0,0.684093,0.620155,11.0
2,admission_conversion,입원 전환,skipped,too_few_labeled_train_or_test train=0 test=0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,generator,seed,target,target_korean,tstr_auc,train_on_real_auc,utility_gap,tstr_accuracy,tstr_f1,status,reason,n_synthetic_train,n_real_test,pos_rate_synthetic_train,pos_rate_real_test
2,PALSYN,41,admission_conversion,입원 전환,NaN,NaN,NaN,NaN,NaN,skipped,too_few_labeled_train_or_test train=0 test=0,NaN,NaN,NaN,NaN
1,PALSYN,41,los_over_4h,체류시간 초과,0.594898,0.639732,0.044834,0.620155,0.765550,ok,,813.0,258.0,0.682657,0.620155
0,PALSYN,41,readmission_30d,재입원 예측,NaN,NaN,NaN,NaN,NaN,skipped,too_few_labeled_train_or_test train=0 test=258,NaN,NaN,NaN,NaN
5,PALSYN,42,admission_conversion,입원 전환,NaN,NaN,NaN,NaN,NaN,skipped,too_few_labeled_train_or_test train=0 test=0,NaN,NaN,NaN,NaN
4,PALSYN,42,los_over_4h,체류시간 초과,0.624362,0.639732,0.015370,0.647287,0.757333,ok,,791.0,258.0,0.664981,0.620155
3,PALSYN,42,readmission_30d,재입원 예측,NaN,NaN,NaN,NaN,NaN,skipped,too_few_labeled_train_or_test train=0 test=258,NaN,NaN,NaN,NaN
8,PALSYN,43,admission_conversion,입원 전환,NaN,NaN,NaN,NaN,NaN,skipped,too_few_labeled_train_or_test train=0 test=0,NaN,NaN,NaN,NaN
7,PALSYN,43,los_over_4h,체류시간 초과,0.385842,0.639732,0.253890,0.620155,0.765550,ok,,812.0,258.0,0.692118,0.620155
6,PALSYN,43,readmission_30d,재입원 예측,NaN,NaN,NaN,NaN,NaN,skipped,too_few_labeled_train_or_test train=0 test=258,NaN,NaN,NaN,NaN
11,PALSYN,44,admission_conversion,입원 전환,NaN,NaN,NaN,NaN,NaN,skipped,too_few_labeled_train_or_test train=0 test=0,NaN,NaN,NaN,NaN


In [11]:

# ============================================================
# 10. Privacy: ML-Leaks adapted MIA with shadow defender posterior
# ============================================================
# Rationale:
# Generators do not expose predict_proba. Therefore, each synthetic log trains a defender.
# The attack observes the defender's posterior on real member/non-member cases.
# Attack training data is obtained from a shadow generator and shadow defender.

def simple_markov_generate(train_log: pd.DataFrame, n_cases: int, seed=RANDOM_STATE) -> pd.DataFrame:
    """A lightweight shadow generator for attack training.
    This is intentionally a surrogate shadow generator, not the target generator.
    """
    rng = np.random.default_rng(seed)
    # Build empirical trace length, start activity, transition, duration distributions.
    traces = []
    durations = []
    deltas = []
    for _, g in train_log.sort_values([CASE_COL,TIME_COL], na_position='last').groupby(CASE_COL):
        acts = g[ACT_COL].astype(str).tolist()
        if len(acts) < 1: continue
        traces.append(acts)
        t = pd.to_datetime(g[TIME_COL], errors='coerce')
        if t.notna().sum() >= 2:
            durations.append((t.max()-t.min()).total_seconds()/3600)
            dt = t.sort_values().diff().dt.total_seconds().dropna()/3600
            deltas.extend([max(float(x), 1/3600) for x in dt if pd.notna(x)])
    if not traces:
        raise ValueError('No traces for shadow generation')
    lengths = np.array([len(x) for x in traces])
    starts = [x[0] for x in traces]
    trans = defaultdict(list)
    for acts in traces:
        for a,b in zip(acts[:-1], acts[1:]):
            trans[a].append(b)
    all_acts = sorted(set([a for tr in traces for a in tr]))
    if not deltas:
        deltas = [0.1]
    rows=[]
    base_time = pd.Timestamp('2100-01-01')
    for i in range(n_cases):
        L = int(rng.choice(lengths))
        L = max(1, L)
        cur = str(rng.choice(starts))
        acts=[cur]
        for _ in range(L-1):
            nxts = trans.get(cur, all_acts)
            cur = str(rng.choice(nxts))
            acts.append(cur)
        ts = base_time + pd.Timedelta(hours=float(i))
        for j,a in enumerate(acts):
            if j > 0:
                ts = ts + pd.Timedelta(hours=float(rng.choice(deltas)))
            rows.append({CASE_COL: f'shadow_syn_{seed}_{i}', ACT_COL: a, TIME_COL: ts})
    return pd.DataFrame(rows)


def fit_defender_get_posteriors(train_cases, query_cases, target, seed=RANDOM_STATE):
    """Train defender on train_cases and return posterior features for query_cases.
    Returns None when unavailable.
    """
    if target not in train_cases.columns or target not in query_cases.columns:
        return None, 'target_missing'
    tr = train_cases[train_cases[target].notna()].copy()
    qu = query_cases[query_cases[target].notna()].copy()
    if len(tr) < 20 or len(qu) < 20:
        return None, 'too_few_labeled'
    if tr[target].nunique() < 2:
        return None, 'single_class_defender_train'
    X_tr, X_qu, feature_cols = align_feature_frames(tr, qu)
    y_tr = tr[target].astype(int).values
    pipe = Pipeline([('prep', make_preprocessor(X_tr)), ('clf', make_predictor(seed))])
    pipe.fit(X_tr, y_tr)
    proba = pipe.predict_proba(X_qu)
    # Ensure binary posterior columns.
    if proba.shape[1] == 1:
        p1 = np.zeros(len(qu)) if pipe.named_steps['clf'].classes_[0] == 0 else np.ones(len(qu))
        proba = np.vstack([1-p1, p1]).T
    elif list(pipe.named_steps['clf'].classes_) == [1,0]:
        proba = proba[:, ::-1]
    p1 = proba[:,1]
    # ML-Leaks-style posterior summary features.
    eps = 1e-12
    feats = pd.DataFrame({
        'p0': proba[:,0],
        'p1': proba[:,1],
        'max_posterior': np.max(proba, axis=1),
        'entropy': -np.sum(proba*np.log(proba+eps), axis=1),
        'margin': np.abs(proba[:,1]-proba[:,0]),
        'true_class_posterior': np.where(qu[target].astype(int).values==1, proba[:,1], proba[:,0]),
    }, index=qu.index)
    return (feats, qu), ''


def run_ml_leaks_mia_for_seed(gen, seed, syn_cases, target):
    """Train shadow attack MLP and evaluate target defender leakage for one synthetic seed."""
    # Need target labels in synthetic and real query sets.
    if target not in syn_cases.columns:
        return {'status':'skipped', 'reason':'target_missing_in_synthetic'}
    if syn_cases[target].notna().sum() < 20 or syn_cases[target].dropna().nunique() < 2:
        return {'status':'skipped', 'reason':'synthetic_target_unavailable_or_single_class'}
    if real_train_cases[target].notna().sum() < 20 or real_test_cases[target].notna().sum() < 20:
        return {'status':'skipped', 'reason':'real_member_nonmember_labels_unavailable'}
    if real_train_cases[target].dropna().nunique() < 2 or real_test_cases[target].dropna().nunique() < 2:
        return {'status':'skipped', 'reason':'real_member_or_nonmember_single_class'}

    # Shadow split from real_train only, disjoint from target eval split by index.
    real_labeled = real_train_raw.copy()
    case_ids = real_train_cases.loc[real_train_cases[target].notna(), CASE_COL].astype(str).values
    if len(case_ids) < 40:
        return {'status':'skipped', 'reason':'too_few_real_cases_for_shadow'}
    sh_member_ids, sh_nonmember_ids = train_test_split(case_ids, test_size=0.5, random_state=seed, shuffle=True)
    shadow_member_log = real_train_raw[real_train_raw[CASE_COL].astype(str).isin(set(sh_member_ids))].copy()
    shadow_nonmember_cases = real_train_cases[real_train_cases[CASE_COL].astype(str).isin(set(sh_nonmember_ids))].copy()
    shadow_member_cases = real_train_cases[real_train_cases[CASE_COL].astype(str).isin(set(sh_member_ids))].copy()

    # Shadow generator and shadow labels. Labels are transferred by case id only when possible;
    # for generated shadow cases, LOS can be derived, other targets will usually be unavailable and skipped.
    try:
        shadow_syn_log = simple_markov_generate(shadow_member_log, n_cases=len(shadow_member_cases), seed=seed)
        shadow_syn_cases = add_real_labels(build_case_base(shadow_syn_log, f'shadow_syn_{seed}'))
        # For clinical labels not derivable from generated log, sample/carry label distribution from shadow_member_cases by case order.
        # This is necessary to train a shadow defender for readmission/admission when synthetic files actually contain labels.
        # It is not used to fabricate target synthetic labels.
        for t in TARGETS:
            if shadow_syn_cases[t].notna().sum() == 0 and shadow_member_cases[t].notna().sum() >= len(shadow_syn_cases):
                vals = shadow_member_cases[t].dropna().sample(n=len(shadow_syn_cases), replace=True, random_state=seed).values
                shadow_syn_cases[t] = vals
    except Exception as e:
        return {'status':'skipped', 'reason':f'shadow_generation_failed: {repr(e)}'}

    # Train shadow defender and get posterior features for member/non-member cases.
    shadow_query = pd.concat([shadow_member_cases, shadow_nonmember_cases], ignore_index=True)
    shadow_query['membership_label'] = [1]*len(shadow_member_cases) + [0]*len(shadow_nonmember_cases)
    shadow_post, reason = fit_defender_get_posteriors(shadow_syn_cases, shadow_query, target, seed=seed)
    if shadow_post is None:
        return {'status':'skipped', 'reason':f'shadow_defender_failed: {reason}'}
    X_attack_train, q_shadow = shadow_post
    y_attack_train = q_shadow['membership_label'].values.astype(int)
    if len(np.unique(y_attack_train)) < 2:
        return {'status':'skipped', 'reason':'attack_train_single_class'}

    attack_model = Pipeline([
        ('scaler', StandardScaler()),
        ('mlp', MLPClassifier(hidden_layer_sizes=(64,32), activation='relu', max_iter=500, random_state=seed, early_stopping=True))
    ])
    attack_model.fit(X_attack_train, y_attack_train)

    # Target defender trained on the actual synthetic seed. Query real_train as members, real_test as non-members.
    target_query = pd.concat([real_train_cases, real_test_cases], ignore_index=True)
    target_query['membership_label'] = [1]*len(real_train_cases) + [0]*len(real_test_cases)
    target_query = target_query[target_query[target].notna()].copy()
    target_post, reason = fit_defender_get_posteriors(syn_cases, target_query, target, seed=seed)
    if target_post is None:
        return {'status':'skipped', 'reason':f'target_defender_failed: {reason}'}
    X_attack_test, q_target = target_post
    y_attack_test = q_target['membership_label'].values.astype(int)
    if len(np.unique(y_attack_test)) < 2:
        return {'status':'skipped', 'reason':'attack_test_single_class'}
    mia_score = attack_model.predict_proba(X_attack_test)[:,1]
    mia_auc = roc_auc_score(y_attack_test, mia_score)
    return {
        'status':'ok',
        'reason':'',
        'mia_auc': float(mia_auc),
        'vulnerable_auc_gt_0_6': bool(mia_auc > MIA_VULNERABLE_AUC),
        'attack_train_n': int(len(y_attack_train)),
        'attack_test_n': int(len(y_attack_test)),
        'attack_train_member_rate': float(np.mean(y_attack_train)),
        'attack_test_member_rate': float(np.mean(y_attack_test)),
    }

privacy_rows = []
for (gen, seed), syn_cases in syn_case_tables.items():
    for target, target_ko in TARGETS.items():
        res = run_ml_leaks_mia_for_seed(gen, seed, syn_cases, target)
        privacy_rows.append({
            'generator': gen,
            'seed': seed,
            'target': target,
            'target_korean': target_ko,
            'mia_auc': res.get('mia_auc', np.nan),
            'vulnerable_auc_gt_0_6': res.get('vulnerable_auc_gt_0_6', np.nan),
            'status': res.get('status'),
            'reason': res.get('reason',''),
            'attack_train_n': res.get('attack_train_n', np.nan),
            'attack_test_n': res.get('attack_test_n', np.nan),
        })

privacy_by_seed = pd.DataFrame(privacy_rows).sort_values(['generator','seed','target'])
privacy_by_seed.to_csv(OUT_DIR/'privacy_ml_leaks_by_seed.csv', index=False)
display(privacy_by_seed)


,generator,seed,target,target_korean,mia_auc,vulnerable_auc_gt_0_6,status,reason,attack_train_n,attack_test_n
2,PALSYN,41,admission_conversion,입원 전환,NaN,NaN,skipped,synthetic_target_unavailable_or_single_class,NaN,NaN
1,PALSYN,41,los_over_4h,체류시간 초과,0.474548,False,ok,,899.0,1157.0
0,PALSYN,41,readmission_30d,재입원 예측,NaN,NaN,skipped,synthetic_target_unavailable_or_single_class,NaN,NaN
5,PALSYN,42,admission_conversion,입원 전환,NaN,NaN,skipped,synthetic_target_unavailable_or_single_class,NaN,NaN
4,PALSYN,42,los_over_4h,체류시간 초과,0.514198,False,ok,,899.0,1157.0
3,PALSYN,42,readmission_30d,재입원 예측,NaN,NaN,skipped,synthetic_target_unavailable_or_single_class,NaN,NaN
8,PALSYN,43,admission_conversion,입원 전환,NaN,NaN,skipped,synthetic_target_unavailable_or_single_class,NaN,NaN
7,PALSYN,43,los_over_4h,체류시간 초과,0.481510,False,ok,,899.0,1157.0
6,PALSYN,43,readmission_30d,재입원 예측,NaN,NaN,skipped,synthetic_target_unavailable_or_single_class,NaN,NaN
11,PALSYN,44,admission_conversion,입원 전환,NaN,NaN,skipped,synthetic_target_unavailable_or_single_class,NaN,NaN


In [12]:

# ============================================================
# 11. Aggregate seed-level results: mean ± std
# ============================================================
def mean_std_table(df, group_cols, metric_cols):
    agg_dict = {}
    for m in metric_cols:
        agg_dict[f'{m}_mean'] = (m, 'mean')
        agg_dict[f'{m}_std'] = (m, 'std')
        agg_dict[f'{m}_n'] = (m, lambda x: int(pd.Series(x).notna().sum()))
    return df.groupby(group_cols).agg(**agg_dict).reset_index()

fidelity_mean_std = mean_std_table(
    fidelity_by_seed,
    ['generator'],
    ['activity_kl_divergence','trace_variant_jaccard','duration_wasserstein_hours']
)
fidelity_mean_std.to_csv(OUT_DIR/'fidelity_mean_std_by_generator.csv', index=False)

utility_mean_std = mean_std_table(
    utility_by_seed[utility_by_seed.status=='ok'].copy(),
    ['generator','target','target_korean'],
    ['tstr_auc','train_on_real_auc','utility_gap','tstr_accuracy','tstr_f1']
)
utility_mean_std.to_csv(OUT_DIR/'utility_mean_std_by_generator_target.csv', index=False)

privacy_mean_std = mean_std_table(
    privacy_by_seed[privacy_by_seed.status=='ok'].copy(),
    ['generator','target','target_korean'],
    ['mia_auc']
)
privacy_mean_std.to_csv(OUT_DIR/'privacy_mean_std_by_generator_target.csv', index=False)

print('Fidelity mean/std')
display(fidelity_mean_std)
print('Utility mean/std')
display(utility_mean_std)
print('Privacy mean/std')
display(privacy_mean_std)


Fidelity mean/std


,generator,activity_kl_divergence_mean,activity_kl_divergence_std,activity_kl_divergence_n,trace_variant_jaccard_mean,trace_variant_jaccard_std,trace_variant_jaccard_n,duration_wasserstein_hours_mean,duration_wasserstein_hours_std,duration_wasserstein_hours_n
0,PALSYN,0.227416,0.014212,5,0.000838,0.000593,5,25824.808028,4999.426137,5
1,ProcessGAN,0.238091,0.362019,5,0.000358,0.000800,5,7.593741,0.039567,5
2,Rule_based,0.000415,0.000279,5,0.034656,0.001187,5,1.711678,0.334593,5


Utility mean/std


,generator,target,target_korean,tstr_auc_mean,tstr_auc_std,tstr_auc_n,train_on_real_auc_mean,train_on_real_auc_std,train_on_real_auc_n,utility_gap_mean,utility_gap_std,utility_gap_n,tstr_accuracy_mean,tstr_accuracy_std,tstr_accuracy_n,tstr_f1_mean,tstr_f1_std,tstr_f1_n
0,PALSYN,los_over_4h,체류시간 초과,0.551722,0.099187,5,0.639732,0.0,5,0.088010,0.099187,5,0.617829,0.019107,5,0.753908,0.012644,5
1,ProcessGAN,los_over_4h,체류시간 초과,0.506020,0.074901,5,0.639732,0.0,5,0.133712,0.074901,5,0.379845,0.000000,5,0.000000,0.000000,5
2,Rule_based,los_over_4h,체류시간 초과,0.451480,0.008793,5,0.639732,0.0,5,0.188253,0.008793,5,0.412403,0.009729,5,0.434630,0.058425,5


Privacy mean/std


,generator,target,target_korean,mia_auc_mean,mia_auc_std,mia_auc_n
0,PALSYN,los_over_4h,체류시간 초과,0.484381,0.017280,5
1,ProcessGAN,los_over_4h,체류시간 초과,0.526178,0.009506,5
2,Rule_based,los_over_4h,체류시간 초과,0.489581,0.019788,5


In [13]:

# ============================================================
# 12. Final paper tables
# ============================================================
# Seed-level integrated long table.
final_by_seed = utility_by_seed.merge(
    privacy_by_seed[['generator','seed','target','mia_auc','vulnerable_auc_gt_0_6','status','reason']],
    on=['generator','seed','target'], how='outer', suffixes=('_utility','_privacy')
).merge(
    fidelity_by_seed,
    on=['generator','seed'], how='left'
)
final_by_seed.to_csv(OUT_DIR/'final_by_seed_all_axes.csv', index=False)

# Mean/std integrated table by generator-target.
final_mean_std = utility_mean_std.merge(
    privacy_mean_std,
    on=['generator','target','target_korean'], how='outer', suffixes=('_utility','_privacy')
).merge(
    fidelity_mean_std,
    on='generator', how='left'
)
final_mean_std.to_csv(OUT_DIR/'final_mean_std_by_generator_target.csv', index=False)

# Compact 3x3 table: each cell = Utility AUC mean±std / MIA AUC mean±std.
def fmt_mean_std(row, mean_col, std_col, n_col=None):
    mean = row.get(mean_col, np.nan)
    std = row.get(std_col, np.nan)
    n = row.get(n_col, np.nan) if n_col else np.nan
    if pd.isna(mean):
        return 'NA'
    if pd.isna(std):
        std = 0.0
    return f'{mean:.3f}±{std:.3f}'

compact_rows=[]
for gen in GENERATORS:
    row = {'generator': gen}
    for target, target_ko in TARGETS.items():
        sub = final_mean_std[(final_mean_std.generator==gen) & (final_mean_std.target==target)]
        if sub.empty:
            row[target_ko] = 'NA'
        else:
            r = sub.iloc[0]
            u = fmt_mean_std(r, 'tstr_auc_mean','tstr_auc_std','tstr_auc_n')
            m = fmt_mean_std(r, 'mia_auc_mean','mia_auc_std','mia_auc_n')
            row[target_ko] = f'{u} / {m}' if u != 'NA' or m != 'NA' else 'NA'
    compact_rows.append(row)
compact_3x3 = pd.DataFrame(compact_rows)
compact_3x3.to_csv(OUT_DIR/'paper_table_3x3_utility_auc_mia_auc.csv', index=False)

# Fidelity compact table.
fidelity_compact = fidelity_mean_std.copy()
fidelity_compact.to_csv(OUT_DIR/'paper_table_fidelity_mean_std.csv', index=False)

print('Final seed-level table')
display(final_by_seed.head(20))
print('Paper 3x3 Utility/MIA table')
display(compact_3x3)
print('Fidelity paper table')
display(fidelity_compact)


Final seed-level table


,generator,seed,target,target_korean,tstr_auc,train_on_real_auc,utility_gap,tstr_accuracy,tstr_f1,status_utility,reason_utility,n_synthetic_train,n_real_test,pos_rate_synthetic_train,pos_rate_real_test,mia_auc,vulnerable_auc_gt_0_6,status_privacy,reason_privacy,activity_kl_divergence,trace_variant_jaccard,duration_wasserstein_hours
0,PALSYN,41,admission_conversion,입원 전환,NaN,NaN,NaN,NaN,NaN,skipped,too_few_labeled_train_or_test train=0 test=0,NaN,NaN,NaN,NaN,NaN,NaN,skipped,synthetic_target_unavailable_or_single_class,0.221271,0.001387,29588.675913
1,PALSYN,41,los_over_4h,체류시간 초과,0.594898,0.639732,0.044834,0.620155,0.765550,ok,,813.0,258.0,0.682657,0.620155,0.474548,False,ok,,0.221271,0.001387,29588.675913
2,PALSYN,41,readmission_30d,재입원 예측,NaN,NaN,NaN,NaN,NaN,skipped,too_few_labeled_train_or_test train=0 test=258,NaN,NaN,NaN,NaN,NaN,NaN,skipped,synthetic_target_unavailable_or_single_class,0.221271,0.001387,29588.675913
3,PALSYN,42,admission_conversion,입원 전환,NaN,NaN,NaN,NaN,NaN,skipped,too_few_labeled_train_or_test train=0 test=0,NaN,NaN,NaN,NaN,NaN,NaN,skipped,synthetic_target_unavailable_or_single_class,0.208383,0.001434,32411.230795
4,PALSYN,42,los_over_4h,체류시간 초과,0.624362,0.639732,0.015370,0.647287,0.757333,ok,,791.0,258.0,0.664981,0.620155,0.514198,False,ok,,0.208383,0.001434,32411.230795
5,PALSYN,42,readmission_30d,재입원 예측,NaN,NaN,NaN,NaN,NaN,skipped,too_few_labeled_train_or_test train=0 test=258,NaN,NaN,NaN,NaN,NaN,NaN,skipped,synthetic_target_unavailable_or_single_class,0.208383,0.001434,32411.230795
6,PALSYN,43,admission_conversion,입원 전환,NaN,NaN,NaN,NaN,NaN,skipped,too_few_labeled_train_or_test train=0 test=0,NaN,NaN,NaN,NaN,NaN,NaN,skipped,synthetic_target_unavailable_or_single_class,0.239477,0.000683,23931.993584
7,PALSYN,43,los_over_4h,체류시간 초과,0.385842,0.639732,0.253890,0.620155,0.765550,ok,,812.0,258.0,0.692118,0.620155,0.481510,False,ok,,0.239477,0.000683,23931.993584
8,PALSYN,43,readmission_30d,재입원 예측,NaN,NaN,NaN,NaN,NaN,skipped,too_few_labeled_train_or_test train=0 test=258,NaN,NaN,NaN,NaN,NaN,NaN,skipped,synthetic_target_unavailable_or_single_class,0.239477,0.000683,23931.993584
9,PALSYN,44,admission_conversion,입원 전환,NaN,NaN,NaN,NaN,NaN,skipped,too_few_labeled_train_or_test train=0 test=0,NaN,NaN,NaN,NaN,NaN,NaN,skipped,synthetic_target_unavailable_or_single_class,0.224584,0.000686,20354.663045


Paper 3x3 Utility/MIA table


,generator,재입원 예측,체류시간 초과,입원 전환
0,ProcessGAN,NA,0.506±0.075 / 0.526±0.010,NA
1,PALSYN,NA,0.552±0.099 / 0.484±0.017,NA
2,Rule_based,NA,0.451±0.009 / 0.490±0.020,NA


Fidelity paper table


,generator,activity_kl_divergence_mean,activity_kl_divergence_std,activity_kl_divergence_n,trace_variant_jaccard_mean,trace_variant_jaccard_std,trace_variant_jaccard_n,duration_wasserstein_hours_mean,duration_wasserstein_hours_std,duration_wasserstein_hours_n
0,PALSYN,0.227416,0.014212,5,0.000838,0.000593,5,25824.808028,4999.426137,5
1,ProcessGAN,0.238091,0.362019,5,0.000358,0.000800,5,7.593741,0.039567,5
2,Rule_based,0.000415,0.000279,5,0.034656,0.001187,5,1.711678,0.334593,5


In [14]:

# ============================================================
# 13. Diagnostics and export zip
# ============================================================
# Diagnostics: missing seeds and skipped targets.
diag_rows=[]
for gen in GENERATORS:
    observed = sorted([s for (g,s) in syn_case_tables if g == gen])
    missing = sorted(set(EXPECTED_SEEDS) - set(observed))
    diag_rows.append({'generator': gen, 'observed_seeds': ','.join(map(str, observed)), 'missing_expected_seeds': ','.join(map(str, missing)), 'n_observed': len(observed)})
seed_diagnostics = pd.DataFrame(diag_rows)
seed_diagnostics.to_csv(OUT_DIR/'seed_diagnostics.csv', index=False)

skip_diagnostics = pd.concat([
    utility_by_seed.assign(axis='utility')[['axis','generator','seed','target','status','reason']],
    privacy_by_seed.assign(axis='privacy')[['axis','generator','seed','target','status','reason']],
], ignore_index=True)
skip_diagnostics = skip_diagnostics[skip_diagnostics.status!='ok'].copy()
skip_diagnostics.to_csv(OUT_DIR/'skipped_experiments_diagnostics.csv', index=False)

print('Seed diagnostics')
display(seed_diagnostics)
print('Skipped experiments')
display(skip_diagnostics.head(50))

# Save notebook-independent metadata.
metadata = {
    'case_col': CASE_COL,
    'activity_col': ACT_COL,
    'time_col': TIME_COL,
    'prefix_events': PREFIX_EVENTS,
    'expected_seeds': EXPECTED_SEEDS,
    'targets': TARGETS,
    'mia_vulnerable_auc_threshold': MIA_VULNERABLE_AUC,
    'notes': [
        'Synthetic clinical labels are not fabricated. Missing labels cause Utility/MIA cells to be skipped.',
        'Fidelity is computed per generator seed and aggregated as mean±std.',
        'Utility uses TSTR: train on synthetic, test on real test.',
        'Privacy uses ML-Leaks adapted: shadow synthetic -> shadow defender posterior -> attack MLP -> target defender posterior.'
    ]
}
with open(OUT_DIR/'run_metadata.json','w',encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2, default=str)

zip_path = Path(str(OUT_DIR) + '.zip')
if zip_path.exists():
    zip_path.unlink()
shutil.make_archive(str(OUT_DIR), 'zip', OUT_DIR)
print('Saved outputs to:', OUT_DIR)
print('Zip:', zip_path)


Seed diagnostics


,generator,observed_seeds,missing_expected_seeds,n_observed
0,ProcessGAN,"41,42,43,44,45",,5
1,PALSYN,"41,42,43,44,45",,5
2,Rule_based,"41,42,43,44,45",,5


Skipped experiments


,axis,generator,seed,target,status,reason
0,utility,PALSYN,41,admission_conversion,skipped,too_few_labeled_train_or_test train=0 test=0
2,utility,PALSYN,41,readmission_30d,skipped,too_few_labeled_train_or_test train=0 test=258
3,utility,PALSYN,42,admission_conversion,skipped,too_few_labeled_train_or_test train=0 test=0
5,utility,PALSYN,42,readmission_30d,skipped,too_few_labeled_train_or_test train=0 test=258
6,utility,PALSYN,43,admission_conversion,skipped,too_few_labeled_train_or_test train=0 test=0
8,utility,PALSYN,43,readmission_30d,skipped,too_few_labeled_train_or_test train=0 test=258
9,utility,PALSYN,44,admission_conversion,skipped,too_few_labeled_train_or_test train=0 test=0
11,utility,PALSYN,44,readmission_30d,skipped,too_few_labeled_train_or_test train=0 test=258
12,utility,PALSYN,45,admission_conversion,skipped,too_few_labeled_train_or_test train=0 test=0
14,utility,PALSYN,45,readmission_30d,skipped,too_few_labeled_train_or_test train=0 test=258


Saved outputs to: results_problemB_seeded_corrected
Zip: results_problemB_seeded_corrected.zip
